# G-EMO Android APK build — Google Colab

Builds the `implementation/g-emo-core` branch of `SE-Emam/G-EMO` using Buildozer. The latest GitHub Actions failure was traced to a pip self-upgrade/import mismatch inside python-for-android (`BuildDependencyInstallError`), so this notebook captures logs and retries after a narrowly scoped workaround if that exact failure occurs.

**No production signing keys or secrets are needed for a debug APK.** A successful build is not the same as a successful on-device test.

In [ ]:
# 1) Install Linux build dependencies
!apt-get -qq update
!apt-get -qq install -y git zip unzip openjdk-17-jdk python3-pip autoconf libtool pkg-config zlib1g-dev libncurses5-dev libncursesw5-dev libtinfo6 cmake libffi-dev libssl-dev automake
%env JAVA_HOME=/usr/lib/jvm/java-17-openjdk-amd64
%env ANDROID_ACCEPT_SDK_LICENSE=1
%env ACCEPT_ANDROID_SDK_LICENSE=1
!java -version
!python --version

In [ ]:
# 2) Get the exact implementation branch
%cd /content
!rm -rf G-EMO
!git clone --depth 1 --branch implementation/g-emo-core https://github.com/SE-Emam/G-EMO.git G-EMO
%cd /content/G-EMO
!git rev-parse HEAD
!test -f buildozer.spec && sed -n '1,160p' buildozer.spec
!python -m pip install --upgrade pip setuptools wheel
!python -m pip install buildozer 'Cython==0.29.36'
!buildozer --version

In [ ]:
# 3) Build; preserve full log even if the first attempt fails
import os, subprocess, pathlib, re, sys, time
os.chdir('/content/G-EMO')
log_path = pathlib.Path('/content/g-emo-build.log')
def run_build(label):
    print(f'===== {label} =====', flush=True)
    with log_path.open('w' if label.endswith('first attempt') else 'a') as log:
        p = subprocess.Popen(['buildozer', '-v', 'android', 'debug'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            print(line, end='')
            log.write(line); log.flush()
        return p.wait()
rc = run_build('Buildozer first attempt')
print('First build exit code:', rc)
if rc != 0:
    log = log_path.read_text(errors='replace')
    if 'BuildDependencyInstallError' in log and 'pip._internal.exceptions' in log:
        print('\nDetected the same pip self-upgrade/import mismatch as GitHub Actions. Applying a narrow retry workaround to the downloaded python-for-android source only.')
        root = pathlib.Path('/content/G-EMO/.buildozer/android/platform/python-for-android')
        changed = []
        if root.exists():
            for f in root.rglob('*.py'):
                try: s = f.read_text()
                except Exception: continue
                if 'pip install -U pip' in s:
                    f.write_text(s.replace('pip install -U pip', 'pip install --upgrade --force-reinstall pip'))
                    changed.append(str(f))
        print('Patched files:', changed if changed else 'No exact self-upgrade command found; no source change made.')
        if changed:
            rc = run_build('Buildozer retry after pip self-upgrade workaround')
            print('Retry exit code:', rc)
    else:
        print('The failure differs from the known GitHub Actions error; see the log tail below.')
if rc != 0:
    print('\n===== LAST 120 LOG LINES =====')
    print('\n'.join(log_path.read_text(errors='replace').splitlines()[-120:]))
else:
    print('Build command completed successfully.')

In [ ]:
# 4) Verify APK outputs and calculate SHA-256
from pathlib import Path
import hashlib
apks = sorted(Path('/content/G-EMO/bin').glob('*.apk')) if Path('/content/G-EMO/bin').exists() else []
if not apks:
    print('BLOCKED: no APK file found. Review /content/g-emo-build.log before retrying.')
else:
    for apk in apks:
        digest = hashlib.sha256(apk.read_bytes()).hexdigest()
        print(f'APK: {apk}\nSize: {apk.stat().st_size:,} bytes\nSHA-256: {digest}\n')
    print('PASS: APK file exists. Emulator/device launch test is still required.')

In [ ]:
# 5) Download APK (run after a successful build)
from google.colab import files
from pathlib import Path
apks = sorted(Path('/content/G-EMO/bin').glob('*.apk')) if Path('/content/G-EMO/bin').exists() else []
if apks:
    files.download(str(apks[0]))
else:
    print('No APK available to download yet.')